# Landmine Victims in Colombia: A Reproducible Data Workflow

**Author:** Travis Carlson
**Dataset:** *Situación Víctimas Minas Antipersonal en Colombia*, published by the Presidencia de la República de Colombia on the national open data portal (CC BY-SA 4.0)

This notebook builds a reproducible workflow for Colombia's official register of people killed or injured by antipersonnel mines, improvised devices that act like them, and unexploded ordnance, from 1990 to 2023 (12,390 records, one per victim). It loads the published file unchanged, cleans it through a documented pipeline, explores who was harmed, where and when, and charts the main patterns. The cleaned table is the base for later projects on humanitarian mine action.

## 1. Setup

Libraries, file locations, the provenance of the raw file, and one chart style used throughout. The workflow uses only Python, NumPy, Pandas, Matplotlib and Seaborn.

In [1]:
import hashlib
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

# Files (the raw file sits in the same folder as this notebook)
DATA_FILE = Path("victimas_minas_antipersonal.csv")
CLEAN_FILE = Path("victimas_minas_antipersonal_clean.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Provenance of the raw file
SOURCE_PAGE = ("https://www.datos.gov.co/Inclusi-n-Social-y-Reconciliaci-n/"
               "Situaci-n-V-ctimas-Minas-Antipersonal-en-Colombia/yhxn-eqqw")
DOWNLOAD_URL = "https://www.datos.gov.co/api/views/yhxn-eqqw/rows.csv?accessType=DOWNLOAD"
DOWNLOADED_ON = "2026-10-18"
EXPECTED_SHA256 = "6ac13df53d0024bc9d96f8b48d7622934b5bd1a334d40cf4155bf4cea804a84d"

# Chart style: light surface, quiet grid, and one fixed colour per group in every chart
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
GROUP_COLOURS = {"Civilian": "#2a78d6", "Security forces": "#eb6834"}
NOT_RECORDED = "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1, "grid.linestyle": "-",
    "axes.axisbelow": True, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "font.size": 10, "lines.linewidth": 2, "legend.frameon": False,
})

for name, module in [("pandas", pd), ("numpy", np), ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{name:<10} {module.__version__}")

pandas     3.0.6
numpy      2.5.4
matplotlib 3.11.2
seaborn    0.13.2


## 2. Ingestion

The raw file is a snapshot of the portal's CSV export, kept unchanged next to this notebook. Its SHA-256 checksum is compared with the one recorded when it was downloaded, so anyone rerunning the notebook can confirm they are using the same data.

In [2]:
def file_sha256(path):
    """Return the SHA-256 checksum of a file, reading it in 1 MB chunks.

    Used to confirm that the raw file is the same snapshot recorded in the README.
    A different checksum means a newer download or a changed file.
    """
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


checksum = file_sha256(DATA_FILE)
print(f"Source page:   {SOURCE_PAGE}")
print(f"Downloaded on: {DOWNLOADED_ON}")
print(f"SHA-256:       {checksum}")
if checksum == EXPECTED_SHA256:
    print("Checksum matches the recorded snapshot.")
else:
    print("WARNING: checksum differs from the recorded snapshot; results may not match the report.")

df_raw = pd.read_csv(DATA_FILE, encoding="utf-8")
print(f"\nLoaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

Source page:   https://www.datos.gov.co/Inclusi-n-Social-y-Reconciliaci-n/Situaci-n-V-ctimas-Minas-Antipersonal-en-Colombia/yhxn-eqqw
Downloaded on: 2026-10-18
SHA-256:       6ac13df53d0024bc9d96f8b48d7622934b5bd1a334d40cf4155bf4cea804a84d
Checksum matches the recorded snapshot.

Loaded 12,390 rows x 18 columns


,departamento,codigodanedepartamento,municipio,codigodanemunicipio,tipoarea,sitio,ano,mes,rangoedad,grupoetnico,condicion,estado,genero,latitudcabecera,longitudcabecera,tipoevento,Ubicación,Actividad
0,ANTIOQUIA,5,GRANADA,5313,Rural,Sin información,2006,1,Mayor de 18 años,no,Civil,Hombre,Herido,6.143703,-75.185183,Sin informacion,"(6.143703, -75.185183)","Pasando, estando cerca"
1,CAQUETA,18,MONTAÑITA,18410,Rural,Sin información,2009,12,Mayor de 18 años,no,Civil,Hombre,Herido,1.494944,-75.463090,Sin informacion,"(1.49494421255, -75.4630895064)",Erradicador manual - PCI
2,BOLIVAR,13,ARENAL,13042,Rural,Sin información,2023,2,Mayor de 18 años,no,Civil,Hombre,Herido,8.471395,-73.942728,Campo,"(8.47139540671, -73.9427278067)",Agricultura
3,BOLIVAR,13,ARENAL,13042,Rural,Sin información,2023,2,Mayor de 18 años,no,Civil,Hombre,Herido,8.471395,-73.942728,Campo,"(8.47139540671, -73.9427278067)",Agricultura
4,PUTUMAYO,86,PUERTO ASÍS,86568,Rural,Sin información,2008,4,Mayor de 18 años,no,Civil,Hombre,Muerto,0.502569,-76.498180,Sin informacion,"(0.502569, -76.49818)",Erradicador manual - PCI


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 12390 entries, 0 to 12389
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   departamento            12390 non-null  str    
 1   codigodanedepartamento  12390 non-null  int64  
 2   municipio               12390 non-null  str    
 3   codigodanemunicipio     12390 non-null  int64  
 4   tipoarea                12390 non-null  str    
 5   sitio                   12390 non-null  str    
 6   ano                     12390 non-null  int64  
 7   mes                     12390 non-null  int64  
 8   rangoedad               12390 non-null  str    
 9   grupoetnico             12390 non-null  str    
 10  condicion               12390 non-null  str    
 11  estado                  12390 non-null  str    
 12  genero                  12390 non-null  str    
 13  latitudcabecera         12390 non-null  float64
 14  longitudcabecera        12390 non-null  float64
 